# 40 — Target Control Extension

Uses the existing `configs/run.yml` as the primary registry.
Optionally merges extra sites from `configs/targets_controls_phase40.yml`.

Outputs:
- `outputs/40_target_control_extension/sites_registry.csv`
- `outputs/40_target_control_extension/manifest.json`

In [ ]:
import os, json, hashlib, platform, sys, math, time
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except pd.errors.EmptyDataError:
        return pd.DataFrame(), {"path": str(p), "status": "empty_data"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {
            "python": sys.version,
            "platform": platform.platform(),
        },
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

cfg = load_yaml(CONFIGS / "run.yml")
run_cfg = cfg.get("run", {})
date_from = run_cfg.get("date_from", "2025-01-01")
date_to = run_cfg.get("date_to", "2025-01-31")
sites = cfg.get("sites", [])
sites_df = pd.DataFrame(sites)
if sites_df.empty:
    sites_df = pd.DataFrame(columns=["id","name","role","lat","lon"])

In [ ]:
step = "40_target_control_extension"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

extra_cfg_path = CONFIGS / "targets_controls_phase40.yml"
extra_cfg = load_yaml(extra_cfg_path)
extra_sites = extra_cfg.get("sites", []) if isinstance(extra_cfg, dict) else []

existing_ids = set(sites_df.get("id", pd.Series(dtype=str)).astype(str).tolist()) if not sites_df.empty else set()
extra_rows = [s for s in extra_sites if str(s.get("id")) not in existing_ids]
registry = pd.concat([sites_df, pd.DataFrame(extra_rows)], ignore_index=True) if extra_rows else sites_df.copy()

if "verified" not in registry.columns:
    registry["verified"] = True
if "source" not in registry.columns:
    registry["source"] = "configs/run.yml"

registry.to_csv(out / "sites_registry.csv", index=False)
display(registry)

manifest = manifest_base(step, [CONFIGS / "run.yml", extra_cfg_path])
manifest["artifacts"].append({"path": str(out / "sites_registry.csv"), "sha256": sha256_file(out / "sites_registry.csv")})
manifest["site_count"] = int(len(registry))
manifest["target_count"] = int((registry["role"] == "target").sum()) if "role" in registry.columns else 0
manifest["control_count"] = int((registry["role"] == "control").sum()) if "role" in registry.columns else 0
write_json(out / "manifest.json", manifest)
print("Wrote:", out)